In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

NUM_CLASSES = 11
BATCH_SIZE = 32
LOCAL_EPOCHS = 15
LR = 1e-4

TRAIN_DIR = "TRAIN"
VAL_DIR = "VAL"
TEST_DIR = "TEST"

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [ ]:
# Loading ResNet-50
def get_model():
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)

# Loading Swin_v2_Base
# def get_model():
#     model = models.swin_v2_b(weights=models.Swin_V2_B_Weights.IMAGENET1K_V1)
#     model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading ViT_b_16
# def get_model():
#     model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
#     model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading ConvNext Base
# def get_model():
#     model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
#     model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading EfficientNet_V2_S
# def get_model():
#     model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
#     model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
#     return model.to(DEVICE)

In [5]:
def get_client_loader(client_name):
    client_path = os.path.join(TRAIN_DIR, client_name)
    print(f"Loading data from {client_path}")
    dataset = datasets.ImageFolder(client_path, transform=train_transform)
    print(f"Number of training data instances: {len(dataset)}")
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)


In [6]:
val_loader = DataLoader(
    datasets.ImageFolder(VAL_DIR, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    datasets.ImageFolder(TEST_DIR, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro')  
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    return acc, precision, recall, f1

In [ ]:
def average_models(models):
    avg_model = copy.deepcopy(models[0])
    avg_dict = avg_model.state_dict()

    for key in avg_dict.keys():

        tensors = [m.state_dict()[key] for m in models]

        if tensors[0].dtype in [torch.float32, torch.float64, torch.float16]:
            avg_dict[key] = torch.stack(tensors, dim=0).mean(dim=0)
        else:
            avg_dict[key] = tensors[0]

    avg_model.load_state_dict(avg_dict)
    return avg_model

def greedy_soup(models, val_loader, val_scores):
    sorted_indices = sorted(range(len(models)), key=lambda i: val_scores[i], reverse=True)
    soup_model = copy.deepcopy(models[sorted_indices[0]])
    best_score = val_scores[sorted_indices[0]]
    selected = [sorted_indices[0]]

    for idx in sorted_indices[1:]:
        temp_model = average_models([soup_model, models[idx]])
        score = evaluate(temp_model, val_loader)[0]

        if score > best_score:
            soup_model = temp_model
            best_score = score
            selected.append(idx)

    return soup_model, selected, val_scores

In [ ]:
def reliability_weighted_avg(models, reliabilities, selected_indices):
    weights = torch.tensor([reliabilities[i] for i in selected_indices])
    weights = weights / weights.sum()

    global_model = copy.deepcopy(models[selected_indices[0]])
    global_dict = global_model.state_dict()

    for key in global_dict.keys():
        weighted_params = torch.stack([
            weights[i] * models[selected_indices[i]].state_dict()[key]
            for i in range(len(selected_indices))
        ], dim=0)

        global_dict[key] = weighted_params.sum(dim=0)

    global_model.load_state_dict(global_dict)
    return global_model

In [ ]:
def simple_avg(models, selected_indices):
    global_model = copy.deepcopy(models[selected_indices[0]])
    global_dict = global_model.state_dict()

    for key in global_dict.keys():
        params = torch.stack([
            models[i].state_dict()[key].float()
            for i in selected_indices
        ], dim=0)

        global_dict[key] = params.mean(dim=0)

    global_model.load_state_dict(global_dict)
    return global_model

In [ ]:
clients = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d)) and not d.startswith(".")
])

print(f"Found clients: {clients}")

In [ ]:
CKPT_DIR = "Checkpoints"

client_models = []
client_val_accs = []
client_val_f1s = []

for client in clients:
    ckpt_path = os.path.join(CKPT_DIR, f"{client}.pth")
    checkpoint = torch.load(ckpt_path)

    model = get_model()
    model.load_state_dict(checkpoint["model_state_dict"])

    client_models.append(model)
    client_val_accs.append(checkpoint["val_accuracy"])
    client_val_f1s.append(checkpoint["val_f1"])
    print(f"Done: {client}")

In [ ]:
for i in range(len(client_val_accs)):
    print(f"Client {i+1}: Acc={client_val_accs[i]:.4f}, F1={client_val_f1s[i]:.4f}")

#### Ablation 1: Only Greedy Selection, No Reliability Weighted Aggregation

In [ ]:
soup_model, selected_indices, reliabilities = greedy_soup(client_models, val_loader, client_val_accs)

global_model = simple_avg(client_models, selected_indices)

acc, precision, recall, f1 = evaluate(global_model, test_loader)
print(f"Global Model Test Accuracy: {acc:.4f}")
print(f"Global Model Test Precision: {precision:.4f}")
print(f"Global Model Test Recall: {recall:.4f}")
print(f"Global Model Test F1 Score: {f1:.4f}")

print("Selected Clients:", selected_indices)

#### Ablation 2: Only Reliability Weighted Aggregation, No Greedy Selection

In [ ]:
def reliability_weighted_no_greedy(models, reliabilities):
    weights = torch.tensor(reliabilities, dtype=torch.float32)
    weights = weights / weights.sum()

    global_model = copy.deepcopy(models[0])
    global_dict = global_model.state_dict()

    for key in global_dict.keys():
        weighted_params = torch.stack([
            weights[i] * models[i].state_dict()[key].float()
            for i in range(len(models))
        ], dim=0)

        global_dict[key] = weighted_params.sum(dim=0)

    global_model.load_state_dict(global_dict)
    return global_model

In [ ]:
soup_model, selected_indices, reliabilities = greedy_soup(client_models, val_loader, client_val_accs)

global_model = reliability_weighted_no_greedy(client_models, reliabilities)

test_acc, test_pre, test_recall, test_f1 = evaluate(global_model, test_loader)
print("Global Model Test Accuracy:", test_acc)
print("Global Model Test Precision:", test_pre)
print("Global Model Test Recall:", test_recall)
print("Global Model Test F1 Score:", test_f1)